# optimizer-class-dispatch — ex2: register a new optimizer at runtime

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-class-dispatch`. Running the final beacon cell reports progress against the `Config: Optimizer class dispatch` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: Optimizer class dispatch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-class-dispatch`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-class-dispatch"
DD_SUBTOPIC = "Config: Optimizer class dispatch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Optimizer dispatch — runtime registration

Ex1 hard-coded three entries in `OPTIMIZER_CLASSES`. The next design step is to let CALLERS register new optimizers at runtime (plugin systems, experiments, custom optimizers from research code):

```python
OPTIMIZER_CLASSES = {
    'sgd':   t.optim.SGD,
    'adam':  t.optim.Adam,
    'adamw': t.optim.AdamW,
}

def register_optimizer(name: str, cls):
    OPTIMIZER_CLASSES[name] = cls
```

**Why mutate the module-level dict.** This is the simplest registry pattern — it lets `register_optimizer('lion', Lion)` from a plugin module take effect everywhere that imports `OPTIMIZER_CLASSES`. No singleton object needed; the dict IS the registry.

**Validation at registration, not at dispatch.** Reject a non-class value at registration time (`isinstance(cls, type)` + subclass of `torch.optim.Optimizer`). This shifts the failure to the plugin loader, where the stack trace points at the bad registration — not later when an unrelated training run dies on `opt.step()`.

**Overwrite-vs-error policy.** Default to OVERWRITE (`d[name] = cls`) so users can intentionally swap implementations (e.g. a fused-CUDA AdamW for the default). If you want strict-add, the caller can check `name in OPTIMIZER_CLASSES` first.

### Exercise 2 — register a new optimizer at runtime

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a runtime-registration function that mutates the module-level dispatch dict so that subsequent construction calls can resolve the new optimizer name without modifying the factory function.
> Keywords: dispatch-table, registry, plugin, registration
> ```

**KCs targeted:** `registry-mutation-by-name`, `registration-time-validation`

Implement TWO things:

1. `ex2_register_optimizer(name, cls)` — mutate the   module-level `OPTIMIZER_CLASSES` dict to add (or   overwrite) the entry `{name: cls}`. Validate that   `cls` is a class AND a subclass of   `torch.optim.Optimizer`; reject anything else with   `TypeError`.
2. `ex2_build_optimizer(name, params, lr)` — the same   factory as ex1 but reading from the (potentially   extended) `OPTIMIZER_CLASSES` registry.

Constraints:
- `OPTIMIZER_CLASSES` must start with the 3 base   entries (`sgd`, `adam`, `adamw`).
- Registration must be IDEMPOTENT for the same `(name,   cls)` pair.
- Overwriting an existing entry is allowed (e.g. swap   in a custom AdamW).
- Unknown name at build time raises `KeyError`.
- Non-class or non-Optimizer-subclass `cls` raises   `TypeError` at REGISTRATION (not at build).

In [ ]:
OPTIMIZER_CLASSES = {
    'sgd':   t.optim.SGD,
    'adam':  t.optim.Adam,
    'adamw': t.optim.AdamW,
}

def ex2_register_optimizer(name, cls):
    if not isinstance(cls, type):
        raise TypeError(
            f'cls must be a class, got {type(cls).__name__}'
        )
    if not issubclass(cls, t.optim.Optimizer):
        raise TypeError(
            f'cls must subclass torch.optim.Optimizer, got {cls.__name__}'
        )
    OPTIMIZER_CLASSES[name] = cls

def ex2_build_optimizer(name, params, lr):
    cls = OPTIMIZER_CLASSES[name]
    return cls(params, lr=lr)


<details><summary>Solution</summary>

```python
OPTIMIZER_CLASSES = {
    'sgd':   t.optim.SGD,
    'adam':  t.optim.Adam,
    'adamw': t.optim.AdamW,
}

def ex2_register_optimizer(name, cls):
    if not isinstance(cls, type):
        raise TypeError(
            f'cls must be a class, got {type(cls).__name__}'
        )
    if not issubclass(cls, t.optim.Optimizer):
        raise TypeError(
            f'cls must subclass torch.optim.Optimizer, got {cls.__name__}'
        )
    OPTIMIZER_CLASSES[name] = cls

def ex2_build_optimizer(name, params, lr):
    cls = OPTIMIZER_CLASSES[name]
    return cls(params, lr=lr)
```

**Validation at registration, not at build.** The registration call is where the bad class entered the system — the stack trace there points at the plugin loader. Validating at build time hides the cause behind whatever training run hits the bad name first.

**Why two `TypeError` branches.** `isinstance(cls, type)` catches non-classes (strings, functions, instances) — `issubclass` would raise its own `TypeError` for those, but the message is cryptic. The explicit guard gives a clear error.

**Idempotence is free.** `d[name] = cls` is idempotent for the same `(name, cls)` pair — re-registration just rewrites the same value. No need for an explicit 'is this already there?' check.

**The registry IS the API.** Plugins can `from your_module import OPTIMIZER_CLASSES` and use the registry directly (e.g. to enumerate available optimizers for a CLI `--help`). That's the win over an `if/elif` chain — the data is inspectable.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()